# Basic Retrieval and Contextual Compression Retrieval

This notebook is organized into categories so that every block of code is easy to follow. The underlying code is unchanged from the original notebook. What has been added is structure, grouping of related cells, and explanations of what each part does and why it matters.

Reference material that the original notebook pointed to.

The LangChain document compressors source code, found in the LangChain GitHub repository under libs/langchain/langchain/retrievers/document_compressors.

The LangChain blog post titled Improving Document Retrieval with Contextual Compression.


## What this notebook demonstrates and why it matters

A plain retrieval system embeds a query and a set of document chunks, then returns the chunks whose embeddings are closest to the query embedding. This works reasonably well, but it has a limitation: an entire chunk is returned even if only one sentence inside it is actually relevant to the question. The rest of that chunk is noise that gets passed on to the language model, taking up context window space and sometimes distracting the model from the truly relevant information.

Contextual compression solves this by adding a second step after retrieval. Instead of returning full chunks as is, a compressor inspects each retrieved chunk together with the query and either extracts only the relevant sentences, drops chunks that are not relevant at all, or removes chunks that are near duplicates of each other. The result is a smaller, more focused set of context that still contains the information needed to answer the question.

This notebook builds up to that idea in stages. It starts with a plain retriever and a plain RetrievalQA chain, then introduces four different kinds of compressors one at a time, so that the effect of each can be understood before moving to the next. It ends by measuring how much shorter the compressed context becomes compared to the original retrieved context, and by combining several compression steps into a single pipeline that feeds a final question answering chain.


## Category 1: Environment Setup

This category installs the packages needed to load documents, split them into chunks, embed them, and store them in a vector database. `langchain_community` provides the document loader and vector store integrations used throughout the notebook, and `faiss cpu` provides Facebook AI Similarity Search, the library used here as the vector database that stores embeddings and performs fast similarity search on a normal CPU.


In [ ]:
!pip install langchain_community

In [ ]:
#facebook ai similarity search
!pip install faiss-cpu

## Category 2: Loading, Splitting, Embedding, and Storing the Document

This category turns a plain text file into a searchable vector index. Four steps happen here in sequence. First the raw text is loaded from disk. Second it is split into smaller chunks so that each chunk is a manageable size for embedding and for later use as context. Third each chunk is converted into a dense vector using an embedding model. Fourth those vectors are stored in a FAISS index and wrapped as a retriever that can be queried with new text.


### Importing the loading, storage, and splitting components

`TextLoader` reads a plain text file into LangChain `Document` objects. `FAISS` is the vector store that holds the embeddings and performs similarity search. `CharacterTextSplitter` breaks a long document into smaller chunks based on a target character count.


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

### Loading the source document

The state of the union address text file is loaded as the knowledge source for this notebook. `TextLoader.load` returns a list containing a single `Document` object, since the loader treats the whole file as one document before any splitting happens.


In [ ]:
documents = TextLoader(
    r"F:\Advance RAG\Data\state_of_the_union.txt",
    encoding="utf-8"
).load()

### Splitting the document into chunks

A chunk size of 500 characters with 100 characters of overlap is used here. The overlap means that some text near a chunk boundary is repeated in the neighboring chunk, which reduces the chance that a sentence gets cut in half in a way that loses its meaning. The two cells below confirm that the splitter returns a plain Python list, and check how many chunks the document was broken into.


In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
texts = text_splitter.split_documents(documents)

In [ ]:
type(texts)

In [ ]:
len(texts)

### Creating embeddings

`HuggingFaceEmbeddings` wraps the `sentence-transformers/all-MiniLM-L6-v2` model, a small and widely used sentence embedding model that turns each text chunk into a dense vector capturing its semantic meaning. This same embedding model is reused several times later in the notebook, including inside one of the compressors.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

### Building the vector store and the baseline retriever

`FAISS.from_documents` embeds every chunk and stores the resulting vectors inside a FAISS index. Calling `as_retriever` turns that index into a retriever object with a simple interface, so that from this point on, retrieving relevant chunks for any query is a single method call away. This `retriever` object is the baseline retrieval mechanism that every compressor introduced later in the notebook will be layered on top of.


In [ ]:
retriever = FAISS.from_documents(texts, embeddings).as_retriever()

## Category 3: Helper Function for Displaying Results

Before running any queries, it helps to have one consistent way to display retrieved documents. The function below takes a list of `Document` objects and prints the content of each one, separated by a divider line, so results are easy to compare visually throughout the rest of the notebook.


In [ ]:
# Helper function for printing docs

def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

## Category 4: Exploring Baseline Retrieval

This category runs three different queries directly against the baseline FAISS retriever, with no compression applied yet. The goal is to see what plain vector similarity search returns on its own, so that the effect of contextual compression can be judged against this starting point later in the notebook.


### Query about a Supreme Court nomination

This query asks about a specific person mentioned in the address. The retrieved chunks are printed to see how closely the top results match the actual topic.


In [ ]:
docs = retriever.invoke("What did the president say about Ketanji Brown Jackson")

In [ ]:
pretty_print_docs(docs)

### Query about top priorities

This query is more open ended, asking about several priorities at once rather than a single named topic. The retrieved chunks are stored separately as `docs2` so that they can be reused later, in particular to measure how much shorter the compressed version of this context becomes.


In [ ]:
docs2 = retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [ ]:
pretty_print_docs(docs2)

### Query about climate change

A third query on a different topic, climate change, is run to see how the retriever behaves on yet another kind of question before any compression is introduced.


In [ ]:
docs3 = retriever.invoke("How did the President propose to tackle the issue of climate change?")


In [ ]:
pretty_print_docs(docs3)

## Category 5: Setting Up the Language Model

Several of the compressors used later in this notebook, and the final question answering chain, need access to a language model to judge relevance or to generate an answer. This category installs the Groq integration, loads an API key from a local environment file, and creates a chat model instance that is reused throughout the rest of the notebook.


In [ ]:
! pip install -U langchain-groq

### Loading the Groq API key

The API key is stored in a local `.env` file and loaded with `python-dotenv` so that it does not need to be hard coded anywhere in the notebook.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Creating the chat model

`ChatGroq` is configured to use the `llama-3.3-70b-versatile` model with temperature set to zero, so that its behavior stays as consistent and factual as possible each time it is called. A quick standalone call to the model, asking it to define RAG, confirms that the connection and credentials work correctly before the model is wired into any retrieval chain.


In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
)

In [ ]:
result = llm.invoke("What is RAG?")

print(result.content)

## Category 6: A Basic RetrievalQA Chain

This category connects the plain baseline retriever directly to the language model using `RetrievalQA`, with no compression step in between. This gives a reference answer to compare against once contextual compression is introduced further down, so the value that compression adds can be judged not just by context length but by the quality of the final answer.


In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [ ]:
query="What were the top three priorities outlined in the most recent State of the Union address?"

In [ ]:
chain.invoke(query)

In [ ]:
print(chain.invoke(query)['result'])

## Category 7: Contextual Compression Using LLMChainExtractor

This category introduces the first compressor, `LLMChainExtractor`. For each retrieved chunk, this compressor asks the language model to extract only the parts of that chunk which are actually relevant to the query, discarding everything else. `ContextualCompressionRetriever` then wraps the baseline retriever together with this compressor, so a single call to `invoke` performs retrieval followed automatically by extraction.

Three example queries are run through this compression retriever below, covering the Supreme Court nomination question, the top priorities question, and the climate change question, so the compressed output can be compared directly against the plain retrieval results from Category 4.


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
compression_retriever=ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

### Extracted content for the Supreme Court nomination query

Compared to the full chunk returned earlier by the plain retriever, the extracted result below should contain only the sentences that are directly relevant to the question about Ketanji Brown Jackson.


In [ ]:
compressed_docs = compression_retriever.invoke("What did the president say about Ketanji Jackson Brown")

In [ ]:
compressed_docs

### Extracted content for the top priorities query

This query is reused from Category 4 so that the compressed result, stored again under the name `compressed_docs`, can be compared directly against the original `docs2` result later when measuring the compression ratio.


In [ ]:
compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [ ]:
compressed_docs

In [ ]:
len(compressed_docs)

In [ ]:
pretty_print_docs(compressed_docs)

### Extracted content for the climate change query

The same climate change question from Category 4 is run through the compression retriever, and the result is stored under a new name, `compressed_docs2`, so it does not overwrite the priorities result above.


In [ ]:
compressed_docs2 = compression_retriever.invoke("How did the President propose to tackle the issue of climate change?")

In [ ]:
pretty_print_docs(compressed_docs2)

## Category 8: Contextual Compression Using LLMChainFilter

`LLMChainFilter` is a lighter alternative to `LLMChainExtractor`. Instead of asking the language model to rewrite or trim each chunk down to only the relevant sentences, it asks a simpler yes or no question for each chunk, whether the chunk is relevant to the query at all, and keeps the chunk whole if the answer is yes or drops it entirely if the answer is no. This is cheaper to run since it avoids generating new text for every chunk, though it does not shrink individual chunks the way extraction does.

Running the same top priorities query through this filter, wrapped in its own `ContextualCompressionRetriever` named `compression_retriever2`, shows how whole chunk filtering compares to the sentence level extraction from Category 7.


In [ ]:
from langchain.retrievers.document_compressors import LLMChainFilter

In [ ]:
filter = LLMChainFilter.from_llm(llm)

In [ ]:
compression_retriever2 = ContextualCompressionRetriever(base_compressor=filter, base_retriever=retriever)

In [ ]:
compressed_docs3 = compression_retriever2.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [ ]:
pretty_print_docs(compressed_docs3)

## Category 9: Measuring How Much Compression Helps

Descriptions of compression are more convincing with numbers attached. This category measures the total character length of the original chunks returned by the plain retriever for the top priorities query, `docs2` from Category 4, against the total character length of the compressed chunks produced by the extractor in Category 7, `compressed_docs`. Dividing the two gives a compression ratio, a simple way to express how much smaller the context handed to the language model has become while still aiming to keep the relevant information.


In [ ]:
original_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(docs2)]))

In [ ]:
original_contexts_len

In [ ]:
compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))

In [ ]:
compressed_contexts_len

In [ ]:
print("Original context length:", original_contexts_len)

In [ ]:
print("Compressed context length:", compressed_contexts_len)

In [ ]:
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")

## Category 10: Contextual Compression Using EmbeddingsFilter

Both compressors used so far call the language model for every single chunk, which adds cost and latency. `EmbeddingsFilter` takes a different, cheaper approach: it embeds the query and each candidate chunk, computes a similarity score between them, and keeps only the chunks whose similarity passes a threshold, all without calling the language model at all. This makes it a fast, low cost filtering step, at the price of being somewhat less precise than a language model based judgment.

The embedding model created earlier in Category 2 is reused here, then wrapped in a fresh `ContextualCompressionRetriever` named `compression_retriever3`, and the same top priorities query is run through it so its output can be compared against the previous two compressors.


In [ ]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
embeddings_filter = EmbeddingsFilter(embeddings=embeddings)

In [ ]:
compression_retriever3 = ContextualCompressionRetriever(base_compressor=embeddings_filter, base_retriever=retriever)

In [ ]:
compressed_docs4 = compression_retriever3.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [ ]:
pretty_print_docs(compressed_docs4)

### Reviewing the compression ratio again

These two print statements reuse the character counts computed earlier in Category 9. They are repeated here so that the original context length and the extractor based compressed context length remain visible right alongside the embeddings filter results above, making it easier to compare the two approaches without scrolling back and forth.


In [ ]:
print("Original context length:", original_contexts_len)

In [ ]:
compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))

In [ ]:
print("Compressed context length:", compressed_contexts_len)

In [ ]:
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")

## Category 11: Combining Multiple Compressors With DocumentCompressorPipeline

Individual compressors each solve one part of the problem. A splitter can break oversized chunks into smaller pieces. A redundant filter can remove chunks that are near duplicates of each other in meaning. A relevance filter can keep only the chunks most similar to the query. `DocumentCompressorPipeline` lets several of these steps run one after another as a single compressor, so a retrieved set of chunks is refined in stages rather than through one single technique.

The pipeline built in this category runs three steps in order. First, `CharacterTextSplitter` breaks the retrieved chunks into smaller pieces of at most 300 characters, splitting on sentence boundaries marked by a period and a space. Second, `EmbeddingsRedundantFilter` removes chunks that are near duplicates of each other based on embedding similarity. Third, `EmbeddingsFilter`, configured here to keep the top 5 chunks, keeps only the pieces most relevant to the query. The three steps are combined into a single `pipeline_compressor` and wrapped in a `ContextualCompressionRetriever`, producing the final compression retriever used for the last part of this notebook.


In [ ]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter


In [ ]:
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0, separator=". ")

In [ ]:
type(splitter)

In [ ]:
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

In [ ]:
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings,
    k=5
)

In [ ]:
pipeline_compressor = DocumentCompressorPipeline(transformers=[splitter, redundant_filter, relevant_filter])

In [ ]:
compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline_compressor, base_retriever=retriever)

### Running the pipeline compressor on the top priorities query

The same top priorities question is sent through this combined pipeline. Printing the length of the result and then the content of each piece shows how the three stage pipeline splits, deduplicates, and filters the retrieved context down to five focused passages.


In [ ]:
compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [ ]:
print(len(compressed_docs))

In [ ]:
pretty_print_docs(compressed_docs)

## Category 12: Making a Chain to Connect With the LLM

This final category assembles the complete pipeline. The Groq chat model is configured again here, in the same way as in Category 5, and connected through `RetrievalQA` to the `compression_retriever` built in Category 11, the retriever that performs splitting, deduplication, and relevance filtering before any text reaches the language model. A single call to `chain.invoke` now performs vector retrieval, three stage compression, and answer generation together, and returns a natural language answer grounded in that compressed context.


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
)

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [ ]:
query="What were the top three priorities outlined in the most recent State of the Union address?"

In [ ]:
chain.invoke(query)

In [ ]:
print(chain.invoke(query)['result'])

### Reading the final answer

The provided context does not explicitly state the top three priorities, but it does mention a Unity Agenda for the Nation with four big things that can be done together, and also mentions other initiatives. Based on the given information, some of the priorities that appear in the retrieved and compressed context include ending the shutdown of schools and businesses and getting Americans back to work, investing in education such as increasing Pell Grants, supporting HBCUs, and community colleges, and beating the opioid epidemic by increasing funding for prevention, treatment, harm reduction, and recovery.


## Conclusion

This notebook worked through the idea of contextual compression retrieval step by step, starting from plain vector search and ending with a multi stage compression pipeline feeding a full question answering chain.

The baseline FAISS retriever, built by embedding text chunks with a sentence transformer model, returned entire chunks based purely on embedding similarity. This is fast and simple, but it often returns more text than is actually needed to answer a question, since only part of a chunk may be relevant while the rest is just accompanying context.

Four different compressors were then explored on top of that same baseline retriever. `LLMChainExtractor` used the language model to rewrite each chunk down to only its relevant sentences, giving the most precise but also the most expensive results since the model has to process every chunk individually. `LLMChainFilter` used the language model more cheaply, deciding only whether to keep or drop each whole chunk rather than rewriting it. `EmbeddingsFilter` avoided calling the language model altogether, relying on embedding similarity scores to decide which chunks to keep, making it the fastest and least expensive of the three. `DocumentCompressorPipeline` combined a text splitter, a redundant chunk filter, and a relevance filter into a single multi stage compressor, showing how several lightweight techniques can be chained together to reach a result that is both smaller and less repetitive than any single technique alone.

Measuring the character length of the original retrieved context against the compressed context showed a meaningful reduction, more than twice as short in the example used here, while still preserving the passages needed to answer the question. Feeding that compressed context into a `RetrievalQA` chain, instead of the raw retrieved chunks, produced answers grounded in a smaller and more focused set of evidence.

The overall takeaway is that plain retrieval and contextual compression retrieval solve different halves of the same problem. Retrieval decides which chunks are worth looking at, and compression decides which parts of those chunks are actually worth keeping. Combining both, especially through a staged pipeline that mixes cheap and precise techniques, produces a context that is shorter, less redundant, and better suited to a language model with limited context window space.
